# D — Huy | Isolation Forest & LOF
**Trạng thái:** thí nghiệm trên Validation, **chưa phải kết quả cuối cùng**. Notebook chạy từ trên xuống dưới; `Class` chỉ dùng để chia phân tầng và chấm điểm, không dùng làm feature của hai mô hình không giám sát. Dataset: `data/raw/creditcard.csv` tại máy cá nhân. Không commit dữ liệu gốc.

## 1. Cài đặt, dữ liệu và kiểm tra chất lượng
Tìm thư mục gốc tự động; không phụ thuộc notebook được mở từ `Data_Mining/` hay `notebooks/`.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import average_precision_score

RANDOM_STATE = 42
TOP_K = 100
FEATURES = [f"V{i}" for i in range(1, 29)]
root = Path.cwd().resolve()
while not (root / "requirements.txt").is_file():
    if root == root.parent:
        raise FileNotFoundError("Không tìm thấy thư mục gốc có requirements.txt")
    root = root.parent
csv_path = root / "data" / "raw" / "creditcard.csv"
if not csv_path.is_file():
    raise FileNotFoundError(f"Cần đặt creditcard.csv tại: {csv_path}")
df = pd.read_csv(csv_path)
required = set(FEATURES + ["Time", "Amount", "Class"])
if not required.issubset(df.columns):
    raise ValueError(f"Thiếu các cột: {sorted(required - set(df.columns))}")
y = df["Class"].astype(int)
print("Dataset:", df.shape, "| Missing:", int(df.isna().sum().sum()), "| Duplicates:", int(df.duplicated().sum()))
print("Phân bố nhãn:\n", y.value_counts().sort_index())
print("Root:", root)

Dataset: (284807, 31) | Missing: 0 | Duplicates: 1081
Phân bố nhãn:
 Class
0    284315
1       492
Name: count, dtype: int64
Root: C:\Users\Admin\Downloads\Data_Mining


## 2. Chia dữ liệu (bản thử nghiệm)
Chia phân tầng 70/15/15. **Cảnh báo:** dòng trùng có thể xuất hiện qua các tập; cần Cường và Mong phê duyệt phương pháp chia hợp lệ trước khi đánh giá Test. Tập Test được giữ nguyên, **không chấm điểm trong notebook bản nháp này**.

In [2]:
indices = np.arange(len(df))
idx_train, idx_temp = train_test_split(indices, test_size=0.30, stratify=y, random_state=RANDOM_STATE)
idx_val, idx_test = train_test_split(idx_temp, test_size=0.50, stratify=y.iloc[idx_temp], random_state=RANDOM_STATE)
for name, idx in [("Train", idx_train), ("Validation", idx_val), ("Test (chưa dùng)", idx_test)]:
    print(f"{name}: {len(idx):,} | fraud: {int(y.iloc[idx].sum())} | rate: {y.iloc[idx].mean():.4%}")

def cross_split_overlap(a, b):
    left = df.iloc[a].drop_duplicates()
    right = df.iloc[b]
    return len(right.merge(left, on=list(df.columns), how="inner"))
print("Trùng Train–Validation:", cross_split_overlap(idx_train, idx_val))
print("Trùng Train–Test:", cross_split_overlap(idx_train, idx_test))
print("Trùng Validation–Test:", cross_split_overlap(idx_val, idx_test))

X_train = df.iloc[idx_train][FEATURES]
X_val = df.iloc[idx_val][FEATURES]
y_val = y.iloc[idx_val].to_numpy()
assert np.isfinite(X_train.to_numpy()).all() and np.isfinite(X_val.to_numpy()).all()

Train: 199,364 | fraud: 344 | rate: 0.1725%
Validation: 42,721 | fraud: 74 | rate: 0.1732%
Test (chưa dùng): 42,722 | fraud: 74 | rate: 0.1732%
Trùng Train–Validation: 207
Trùng Train–Test: 249
Trùng Validation–Test: 61


## 3. Hàm chấm điểm chung
Điểm bất thường được đổi dấu để **cao hơn = bất thường hơn**. Precision@100 ở đây tính trên **toàn bộ Validation**, chưa phải ngân sách 100 giao dịch/ngày. AP là `average_precision_score`, không phải diện tích PR nội suy hình thang.

In [3]:
def build_results(indices, labels, scores):
    scores = np.asarray(scores)
    if len(indices) != len(labels) or len(labels) != len(scores):
        raise ValueError("Kích thước indices, labels và scores không khớp")
    out = pd.DataFrame({"original_index": indices, "anomaly_score": scores, "is_fraud": labels})
    out = out.sort_values(["anomaly_score", "original_index"], ascending=[False, True]).reset_index(drop=True)
    out["rank"] = np.arange(1, len(out) + 1)
    return out

def evaluate(results, name, k=TOP_K):
    k = min(k, len(results))
    tp = int(results.head(k)["is_fraud"].sum())
    fraud = int(results["is_fraud"].sum())
    return {"Model": name, "TP": tp, "FP": k - tp, "FN": fraud - tp,
            "Precision@100": tp / k, "Recall@100": tp / fraud if fraud else float("nan"),
            "AP": average_precision_score(results["is_fraud"], results["anomaly_score"])}

## 4. Isolation Forest — không giám sát
Huấn luyện với tất cả mẫu Train; scaler chỉ `fit` trên Train. Dùng V1–V28 để giữ lại cấu hình thí nghiệm gốc của Huy.

In [4]:
iso_demo = Pipeline([
    ("scaler", StandardScaler()),
    ("model", IsolationForest(n_estimators=100, contamination="auto", random_state=RANDOM_STATE, n_jobs=-1))
])
iso_demo.fit(X_train)
iso_scores = -iso_demo.decision_function(X_val)
val_results = build_results(idx_val, y_val, iso_scores)
print("=== ISOLATION FOREST ===")
display(pd.DataFrame([evaluate(val_results, "Isolation Forest")]))

=== ISOLATION FOREST ===


,Model,TP,FP,FN,Precision@100,Recall@100,AP
0,Isolation Forest,24,76,50,0.24,0.324324,0.183223


## 5. LOF — không giám sát, chọn mẫu Train không dùng `Class`
LOF `novelty=True` để chấm điểm các giao dịch mới (Validation). Giữ cố định mẫu Train 20.000 giao dịch khi thử `n_neighbors` để so sánh tham số công bằng. Không chọn mẫu bình thường bằng nhãn cho mô hình chính.

In [5]:
rng = np.random.default_rng(RANDOM_STATE)
sample_idx_unsupervised = rng.choice(idx_train, size=min(20000, len(idx_train)), replace=False)
X_train_lof = df.iloc[sample_idx_unsupervised][FEATURES]
lof_scaler = StandardScaler()
X_train_lof_scaled = lof_scaler.fit_transform(X_train_lof)
X_val_lof_scaled = lof_scaler.transform(X_val)

lof_tuning_rows = []
lof_score_by_n = {}
for neighbors in [10, 20, 50, 100]:
    model = LocalOutlierFactor(n_neighbors=neighbors, novelty=True, n_jobs=-1)
    model.fit(X_train_lof_scaled)
    scores = -model.decision_function(X_val_lof_scaled)
    lof_score_by_n[neighbors] = scores
    result = build_results(idx_val, y_val, scores)
    metrics = evaluate(result, f"LOF n={neighbors}")
    lof_tuning_rows.append({"n_neighbors": neighbors, "Fraud Top 100": metrics["TP"],
                            "Precision@100": metrics["Precision@100"],
                            "Recall@100": metrics["Recall@100"], "AP": metrics["AP"]})
tuning_df = pd.DataFrame(lof_tuning_rows)
display(tuning_df)

best_n = int(tuning_df.sort_values(["Precision@100", "AP"], ascending=False).iloc[0]["n_neighbors"])
lof_results_best = build_results(idx_val, y_val, lof_score_by_n[best_n])
print("Best n_neighbors trong 4 giá trị đã thử:", best_n)

,n_neighbors,Fraud Top 100,Precision@100,Recall@100,AP
0,10,3,0.03,0.040541,0.005698
1,20,3,0.03,0.040541,0.013411
2,50,31,0.31,0.418919,0.191042
3,100,39,0.39,0.527027,0.326613


Best n_neighbors trong 4 giá trị đã thử: 100


## 6. Chẩn đoán phụ: LOF dùng nhãn Train để chọn mẫu bình thường
Phần này tái lập thí nghiệm khám phá ban đầu, **không được xem là LOF hoàn toàn không giám sát**. Không dùng kết quả này làm mô hình chính hoặc thay thế kết quả Test.

In [6]:
normal_idx = idx_train[y.iloc[idx_train].to_numpy() == 0]
normal_sample_idx = np.random.default_rng(RANDOM_STATE).choice(normal_idx, size=min(20000, len(normal_idx)), replace=False)
lof_normal_only = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LocalOutlierFactor(n_neighbors=20, novelty=True, n_jobs=-1))
])
lof_normal_only.fit(df.iloc[normal_sample_idx][FEATURES])
label_filtered_results = build_results(idx_val, y_val, -lof_normal_only.decision_function(X_val))
display(pd.DataFrame([evaluate(label_filtered_results, "LOF lọc nhãn train (tham khảo)")]))

,Model,TP,FP,FN,Precision@100,Recall@100,AP
0,LOF lọc nhãn train (tham khảo),49,51,25,0.49,0.662162,0.421065


## 7. Random Baseline — 100 lần
Dùng 100 seed để ghi lại dao động của baseline ngẫu nhiên; không dựa vào một lần bốc mẫu duy nhất.

In [7]:
baseline_rows = []
for seed in range(100):
    random_scores = np.random.default_rng(seed).random(len(y_val))
    baseline_result = build_results(idx_val, y_val, random_scores)
    metrics = evaluate(baseline_result, "Random baseline")
    baseline_rows.append({"seed": seed, "fraud_top100": metrics["TP"],
                          "precision_at_100": metrics["Precision@100"],
                          "recall_at_100": metrics["Recall@100"], "ap": metrics["AP"]})
baseline_100 = pd.DataFrame(baseline_rows)
baseline_summary = baseline_100[["fraud_top100", "precision_at_100", "recall_at_100", "ap"]].agg(["mean", "std"])
display(baseline_summary)
print("Phân bố fraud_top100:\n", baseline_100["fraud_top100"].value_counts().sort_index())

,fraud_top100,precision_at_100,recall_at_100,ap
mean,0.15000,0.001500,0.002027,0.001959
std,0.35887,0.003589,0.004850,0.000518


Phân bố fraud_top100:
 fraud_top100
0    85
1    15
Name: count, dtype: int64


## 8. Error Analysis và bảng so sánh
TP/FP/FN được định nghĩa **theo ngưỡng Top 100**. Không nhầm với nhãn phân lớp của model. Bảng chính chỉ so sánh hai mô hình không giám sát.

In [8]:
comparison_errors = pd.DataFrame([
    evaluate(val_results, "Isolation Forest"),
    evaluate(lof_results_best, f"LOF (n={best_n})")
])
display(comparison_errors)
for name, result in [("Isolation Forest", val_results), (f"LOF (n={best_n})", lof_results_best)]:
    print(f"{name}: TP={int(result.head(TOP_K)['is_fraud'].sum())}, "
          f"FP={TOP_K - int(result.head(TOP_K)['is_fraud'].sum())}, "
          f"FN={int(result['is_fraud'].sum()) - int(result.head(TOP_K)['is_fraud'].sum())}")

,Model,TP,FP,FN,Precision@100,Recall@100,AP
0,Isolation Forest,24,76,50,0.24,0.324324,0.183223
1,LOF (n=100),39,61,35,0.39,0.527027,0.326613


Isolation Forest: TP=24, FP=76, FN=50
LOF (n=100): TP=39, FP=61, FN=35


## 9. Xuất các file nháp
Xuất CSV trong `outputs/`. Không xuất nhãn Test hoặc giả lập score Test. Những file dưới đây vẫn có hậu tố `_draft` để tránh bị hiểu là kết quả chính thức.

In [9]:
output_dir = root / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)
tuning_df.to_csv(output_dir / "lof_tuning_validation_draft.csv", index=False)
baseline_100.to_csv(output_dir / "random_baseline_100_runs_draft.csv", index=False)
baseline_summary.to_csv(output_dir / "random_baseline_summary_draft.csv")
comparison_errors.to_csv(output_dir / "model_error_analysis_draft.csv", index=False)
print("Đã lưu 4 file đánh giá nháp vào:", output_dir)

Đã lưu 4 file đánh giá nháp vào: C:\Users\Admin\Downloads\Data_Mining\outputs


## 10. Kết luận và điều kiện hoàn thành
- Đây là thí nghiệm sơ bộ, dữ liệu Validation có dòng trùng với Train. **Không kết luận kết quả cuối cùng từ các con số này.**
- LOF sử dụng 20.000 mẫu train, Isolation Forest sử dụng toàn bộ train nên so sánh chưa hoàn toàn đồng nhất.
- `V1–V28` mới là đặc trưng thử nghiệm; chưa tích hợp `Time` và `Amount` đã xử lý bởi Cường.
- Top 100 trên Validation chưa tương ứng câu hỏi nghiệp vụ 100 giao dịch **mỗi ngày**.
- **Việc cần làm sau khi Cường/Mong thống nhất pipeline:** tách train/validation/test không bị leakage, huấn luyện lại, chọn tham số trên Validation, dùng Test đúng một lần để báo cáo, xuất `anomaly_scores_test.csv`, cập nhật decision log.

**Thử tái lập:** trong VS Code chọn **Restart Kernel and Run All Cells**. File này được tổ chức để các ô có thể chạy theo thứ tự, nhưng cần có `creditcard.csv` trên máy; chưa được xác minh bằng lần chạy end-to-end trong môi trường tạo file.